# 10.1 Tips frequency table

Two versions of the same table - one for treated labs only, one for every lab -
showing each energy-saving tip generated by the calculator, the equipment type it
applies to, the number of distinct labs shown that tip, and that count as a share of
only the labs that actually own that equipment type (e.g. "23/34" for a freezer tip,
where 34 is the number of labs in the sample that have a freezer at all - not the
sample size). The calculator substitutes lab-specific figures into a small set of tip
templates (e.g. "...you can save X kWh per year"); numeric slots that are actually
fixed policy constants (e.g. "-70 degrees" for ULTs) are shown as-is rather than
masked. A lab shown the same tip across multiple equipment "types" (e.g. type 1/2/3
fridges) is counted once, not per-type. Both versions are closed with "Any tip
displayed" and "No tips displayed" rows summarizing labs whose calculator did and
didn't show any tips at all.

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_tips_frequency_table import (
    build_tip_frequency_data,
    compute_equipment_ownership,
    make_tips_frequency_table,
)

In [2]:
# Load data
tips = pd.read_csv(config.CLEAN_DATA / "tips_cleaned.csv")
labs = pd.read_csv(config.CLEAN_DATA / "final_dataset.csv")
panel = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_5.csv",
    keep_default_na=False,
    na_values=[""],
)

# List of treated labgroupids
treated_labgroupids = labs.loc[labs["treated"] == 1, "labgroupid"].unique()

# equipment name mapping
equipment_map = {
    "CO2 incubator": "incubator",
    "Freezer": "freezer",
    "Fridge": "fridge",
    "Glassware drying cabinet": "glassware",
    "Microbiological safety cabinet": "microbio",
    "ULT freezer": "ult",
    "Water bath": "bath",
}

## (1) Build and save a table per sample

In [3]:
# Samples: "treated" = only labs that were treated, "all" = all labs
samples = {
    "treated": tips[tips["labgroupid"].isin(treated_labgroupids)],
    "all": tips,
}

# Where to save tables
out_dir = config.OUTPUT / "12_Tips_Tables"
out_dir.mkdir(parents=True, exist_ok=True)

# Build and save tables
tables = {}
for name, sample_df in samples.items():
    freq_df, any_tips_count, no_tips_count = build_tip_frequency_data(sample_df, reference_df=tips)

    # Share of using groups: same sample's labgroupids as the denominator, so e.g. the
    # "treated" table reads "23 of the treated labs that use a freezer", not "23 of
    # all labs" or "23 of the labs that use a freezer across treatment status"
    owners = compute_equipment_ownership(panel, sample_df["labgroupid"].unique(), equipment_map)
    freq_df["n_equipment_labs"] = freq_df["equipment"].map(owners)

    tables[name] = make_tips_frequency_table(freq_df, any_tips_count, no_tips_count)

    table_path = out_dir / f"tips_frequency_table_{name}.tex"
    _ = table_path.write_text(tables[name])

    print(f"[{name}] {len(freq_df)} distinct (tip, equipment) combinations, "
          f"{any_tips_count} labs with any tip displayed, {no_tips_count} labs with no tips displayed")

[treated] 15 distinct (tip, equipment) combinations, 35 labs with any tip displayed, 35 labs with no tips displayed
[all] 17 distinct (tip, equipment) combinations, 67 labs with any tip displayed, 71 labs with no tips displayed


## (2) Preview tables

In [4]:
print(tables["treated"])

\begin{tabular}{@{}>{\raggedright\arraybackslash}p{18cm}>{\raggedright\arraybackslash}p{4.5cm}>{\centering\arraybackslash}p{2.2cm}>{\centering\arraybackslash}p{2.5cm}}
\hline
\addlinespace[0.2cm]
Tip & Equipment & Research groups & Share of using groups \\
\hline
\addlinespace[0.2cm]
Don't let that icing get out of hand! If you can de-ice your freezer (once every 6-12 months) you will keep it energy efficient & Freezer & $23$ & $60.5\%$ \\
\addlinespace[0.15cm]
You're opening the door a lot! Have you used a fridge map or inventory to help you find your stuff? If you can reduce your door openings to 8 per day you can save X kWh per year & Fridge & $20$ & $47.6\%$ \\
\addlinespace[0.15cm]
This unit is on a lot! If you only used this 255 days per year you would save X kWh per year & CO2 incubator & $16$ & $100.0\%$ \\
\addlinespace[0.15cm]
You're opening the door a lot! Have you used a freezer map or inventory to help you find your stuff? If you can reduce your door openings to X per day 

In [5]:
print(tables["all"])

\begin{tabular}{@{}>{\raggedright\arraybackslash}p{18cm}>{\raggedright\arraybackslash}p{4.5cm}>{\centering\arraybackslash}p{2.2cm}>{\centering\arraybackslash}p{2.5cm}}
\hline
\addlinespace[0.2cm]
Tip & Equipment & Research groups & Share of using groups \\
\hline
\addlinespace[0.2cm]
Don't let that icing get out of hand! If you can de-ice your freezer (once every 6-12 months) you will keep it energy efficient & Freezer & $39$ & $53.4\%$ \\
\addlinespace[0.15cm]
You're opening the door a lot! Have you used a fridge map or inventory to help you find your stuff? If you can reduce your door openings to 8 per day you can save X kWh per year & Fridge & $37$ & $43.5\%$ \\
\addlinespace[0.15cm]
This unit is on a lot! If you only used this 255 days per year you would save X kWh per year & CO2 incubator & $28$ & $96.6\%$ \\
\addlinespace[0.15cm]
If you can warm up to -20C you can save X kWh per year & Freezer & $24$ & $32.9\%$ \\
\addlinespace[0.15cm]
You're opening the door a lot! Have you used